# Library OOP Practical Assignment 1

**NAME:** AJAK DENG GARANG DENG  
**REG_NO:** M25B38/031  
**COURSE:** BSDS  
**YEAR:** 2:1  

This notebook keeps the library-management task and demonstrates the four main OOP concepts, validation, and Python lists, tuples, dictionaries, and sets. Comments are included throughout the code to explain what each line does.

## QN.1 – Classes, attributes, methods, and OOP concepts

- **Encapsulation:** `Book` stores its ISBN in the private attribute `__isbn` and checks values before accepting them.
- **Inheritance:** `Book` inherits from the abstract parent class `LibraryItem`; `ReferenceBook` inherits from `Book`; `Student` inherits from `Person`.
- **Abstraction:** `LibraryItem` defines the method `get_details()` that child classes must implement.
- **Polymorphism:** `Book` and `ReferenceBook` provide different versions of `get_details()`.
- **Validation:** The program checks names, ISBNs, dates, copy IDs, and whether a book is available.
- **List:** stores student loans and physical book copies.
- **Tuple:** stores fixed library rules.
- **Dictionary:** stores the catalogue using ISBN as the key.
- **Set:** stores unique physical copy IDs.

In [ ]:
from abc import ABC, abstractmethod  # Import tools used to create an abstract parent class.
from datetime import date  # Import date so the program can record borrowing and return dates.

FINE_PER_DAY = 1000  # Set the fine to UGX 1,000 for each day a book is late.
LIBRARY_RULES = (14, FINE_PER_DAY)  # Store fixed rules as a tuple: loan period in days and fine per day.
catalogue = {}  # Create a dictionary to store books, using each ISBN as a key.
registered_copy_ids = set()  # Create a set to keep physical copy IDs unique.


class LibraryItem(ABC):  # Define an abstract parent class for items in the library.
    def __init__(self, title, author):  # Set up the information shared by library items.
        if not isinstance(title, str) or not title.strip():  # Check that the title is a non-empty string. becuase no book can be stored without a title
            raise ValueError("Book title cannot be empty.")  # Stop the program from accepting an empty title.
        if not isinstance(author, str) or not author.strip():  # Check that the author is a non-empty string. becuase no book can be stored without a author's name.
            raise ValueError("Author name cannot be empty.")  # Stop the program from accepting an empty author.
        self.title = title.strip()  # Store the title without extra spaces.
        self.author = author.strip()  # Store the author without extra spaces.

    @abstractmethod  # Require child classes must implement this method.
    def get_details(self):  # Define the common method for displaying item details.
        pass  # Leave the implementation to each child class.


class Book(LibraryItem):  # Book inherits common title and author features from LibraryItem.
    def __init__(self, isbn, title, author):  # Create a book with an ISBN, title, and author.
        super().__init__(title, author)  # Call the parent class to validate and store title and author for free instead of writing them again.
        self.__isbn = self.validate_isbn(isbn)  # Store ISBN privately after checking it such that people cannot modify it directly.
        self.copies = []  # Use a list to store the physical copies of this book.
        if self.__isbn in catalogue:  # Check whether this ISBN has already been registered.
            raise ValueError("A book with this ISBN is already registered.")  # Reject duplicate ISBNs.
        catalogue[self.__isbn] = self  # Add this book to the catalogue dictionary.

    @staticmethod  # This method does not need to use self or cls. it is just a hepler tool.
    def validate_isbn(isbn):  # Check the format of an ISBN.
        if not isinstance(isbn, str):  # Check that the ISBN was supplied as text.
            raise TypeError("ISBN must be entered as text.")  # Explain the wrong data type.
        clean_isbn = isbn.replace("-", "").replace(" ", "")  # Remove hyphens and spaces before checking.
        if not clean_isbn.isdigit() or len(clean_isbn) not in (10, 13):  # Require 10 or 13 digits. old books have 10 digits, while newer books have 13 digits.
            raise ValueError("ISBN must contain 10 or 13 digits.")  # Report an invalid ISBN.
        return clean_isbn  # Return the cleaned ISBN if it is valid.

    @property  # Allow the private ISBN to be read through a controlled property.
    def isbn(self):  # Define a safe way to read the ISBN.
        return self.__isbn  # Return the private ISBN value.

    def get_details(self):  # Implement the abstract method from LibraryItem.
        return f"Book: {self.title} by {self.author} (ISBN: {self.isbn})"  # Return book information.

    @classmethod  # Let the class return all titles stored in the catalogue.
    def all_titles(cls):  # Define a method to list book titles.
        return [book.title for book in catalogue.values()]  # Build and return a list of titles.

    def __str__(self):  # Set how a book is displayed as text.
        return self.get_details()  # Reuse the details method to avoid repeating code.


class ReferenceBook(Book):  # ReferenceBook inherits all Book features but changes some behaviour.
    def get_details(self):  # Override the parent method to demonstrate polymorphism.
        return f"Reference book: {self.title} by {self.author} (ISBN: {self.isbn})"  # Return a different description.

    def can_be_borrowed(self):  # Define a rule for reference books.
        return False  # Reference books must remain inside the library.


class Person(ABC):  # Create an abstract parent class for people who use the library.
    def __init__(self, name, reg_no, contact):  # Set up details shared by library users.
        if not isinstance(name, str) or not name.strip():  # Check that the name is not empty.
            raise ValueError("Name cannot be empty.")  # Reject an empty name.
        if not isinstance(reg_no, str) or not reg_no.strip():  # Check that the registration number is not empty.
            raise ValueError("Registration number cannot be empty.")  # Reject a missing registration number.
        if not isinstance(contact, str) or not contact.strip():  # Check that contact information is supplied.
            raise ValueError("Contact cannot be empty.")  # Reject missing contact information.
        self.name = name.strip()  # Store the cleaned name.
        self.reg_no = reg_no.strip()  # Store the cleaned registration number.
        self.contact = contact.strip()  # Store the cleaned contact information.

    @abstractmethod  # Require every child class to define its role.
    def get_role(self):  # Define the abstract role method.
        pass  # Leave the role to the child class.


class Student(Person):  # Student inherits name, registration number, and contact from Person.
    def __init__(self, name, reg_no, contact):  # Create a student account.
        super().__init__(name, reg_no, contact)  # Reuse validation and storage from Person.
        self.loans = []  # Use a list to store the student's loans.

    def get_role(self):  # Implement the abstract role method.
        return "Student"  # Return the user's role.

    def add_loan(self, loan):  # Add a loan to the student's list.
        self.loans.append(loan)  # Store the loan so it can be viewed later.

    def list_loans(self):  # Return all loans belonging to this student.
        return self.loans  # Give back the list of loans.

    def __str__(self):  # Set a readable display for the student.
        return f"{self.reg_no} - {self.name} ({self.get_role()})"  # Return student details.


class BookCopy:  # Represent one physical copy of a book.
    def __init__(self, copy_id, book):  # Create a physical copy linked to a book.
        if not isinstance(copy_id, str) or not copy_id.strip():  # Check that the copy ID is valid text.
            raise ValueError("Copy ID cannot be empty.")  # Reject an empty copy ID.
        if copy_id in registered_copy_ids:  # Check whether the ID has already been used.
            raise ValueError("This copy ID already exists.")  # Reject duplicate physical copy IDs.
        if not isinstance(book, Book):  # Ensure the copy belongs to a Book object.
            raise TypeError("A book copy must be linked to a Book.")  # Reject an invalid book object.
        self.copy_id = copy_id.strip()  # Store the copy ID.
        self.book = book  # Store the book associated with this copy.
        self.is_available = True  # A new physical copy is available.
        book.copies.append(self)  # Add this copy to the book's list of copies.
        registered_copy_ids.add(self.copy_id)  # Add the ID to the set to keep IDs unique.

    def mark_borrowed(self):  # Mark the copy as borrowed.
        if not self.is_available:  # Check whether someone already borrowed the copy.
            raise ValueError("This book copy is already borrowed.")  # Report the validation error.
        self.is_available = False  # Update the copy's status to unavailable.

    def mark_returned(self):  # Mark the copy as returned.
        self.is_available = True  # Make the copy available again.

    def __str__(self):  # Define how a physical copy is displayed.
        status = "Available" if self.is_available else "Borrowed"  # Convert the status into readable text.
        return f"{self.copy_id}: {self.book.title} ({status})"  # Return the copy details.

    def __repr__(self):  # Define a useful representation for programmers.
        return f"BookCopy({self.copy_id!r}, {self.book.title!r})"  # Return a short technical description.


class Loan:  # Connect one student to one borrowed physical copy.
    def __init__(self, student, copy, borrow_date, due_date):  # Create a loan record.
        if not isinstance(student, Student):  # Check that a real Student object was provided.
            raise TypeError("Loan must belong to a Student.")  # Reject an invalid student object.
        if not isinstance(copy, BookCopy):  # Check that a real BookCopy object was provided.
            raise TypeError("Loan must refer to a BookCopy.")  # Reject an invalid copy object.
        if not isinstance(borrow_date, date) or not isinstance(due_date, date):  # Check date types.
            raise TypeError("Borrow date and due date must be date objects.")  # Explain the required date type.
        if due_date < borrow_date:  # Check that the due date is not before the borrowing date.
            raise ValueError("Due date cannot be before borrow date.")  # Reject an impossible date range.
        self.student = student  # Store the student who borrowed the book.
        self.copy = copy  # Store the borrowed physical copy.
        self.borrow_date = borrow_date  # Store the date the book was borrowed.
        self.due_date = due_date  # Store the date the book should be returned.
        self.return_date = None  # Use None to show that the book has not yet been returned.

    def return_book(self, return_date):  # Record that the book has been returned.
        if not isinstance(return_date, date):  # Check that the return date is a date object.
            raise TypeError("Return date must be a date object.")  # Reject an invalid date value.
        if self.return_date is not None:  # Check whether the book was already returned.
            raise ValueError("This loan has already been returned.")  # Prevent returning the same loan twice.
        if return_date < self.borrow_date:  # Check that the return date is not before borrowing.
            raise ValueError("Return date cannot be before borrow date.")  # Reject an impossible return date.
        self.return_date = return_date  # Save the return date.
        self.copy.mark_returned()  # Make the physical copy available again.
        return self.calculate_fine()  # Calculate and return any late fine.

    def calculate_fine(self):  # Calculate the fine for a late return.
        if self.return_date is None:  # Check whether the book is still out.
            return 0  # Do not calculate a fine before return in this simple example.
        late_days = max(0, (self.return_date - self.due_date).days)  # Count late days but never use a negative number.
        return late_days * FINE_PER_DAY  # Multiply late days by the daily fine.

    def __str__(self):  # Set a readable description of the loan.
        returned = self.return_date if self.return_date else "Not returned"  # Show the return date or current status.
        return f"{self.student.name} borrowed {self.copy.book.title}; due {self.due_date}; returned: {returned}"  # Return loan details.


def borrow_book(student, copy, borrow_date, due_date):  # Borrow a copy and create its loan record.
    if not isinstance(student, Student) or not isinstance(copy, BookCopy):  # Validate the objects supplied.
        raise TypeError("Please provide a Student and a BookCopy.")  # Explain the required object types.
    if isinstance(copy.book, ReferenceBook):  # Check if the selected book is a reference book.
        raise ValueError("Reference books cannot be borrowed.")  # Apply the reference-book rule.
    if not copy.is_available:  # Check whether another student has borrowed the copy.
        raise ValueError("This book copy is already borrowed.")  # Reject borrowing an unavailable copy.
    loan = Loan(student, copy, borrow_date, due_date)  # Create the loan after validation succeeds.
    copy.mark_borrowed()  # Mark the copy as unavailable.
    student.add_loan(loan)  # Save the loan in the student's list.
    return loan  # Return the new loan so it can be used by the caller.


def return_book(loan, return_date):  # Return a borrowed book and calculate its fine.
    if not isinstance(loan, Loan):  # Check that the supplied object is a loan.
        raise TypeError("Please provide a valid Loan object.")  # Reject the wrong object type.
    fine = loan.return_book(return_date)  # Record the return and calculate the fine.
    print(f"Returned '{loan.copy.book.title}' by {loan.student.name}.")  # Confirm the return.
    print(f"Fine: UGX {fine:,}")  # Display the fine in a readable format.
    return fine  # Return the fine for further use.


## QN.2 – Demonstrate the classes and inheritance

The next cell creates a normal book, a reference book, physical copies, and a student. It also shows polymorphism: both book types use `get_details()`, but the output is different.

In [ ]:
# Create a normal book and register it in the catalogue dictionary.
book1 = Book("9780000001", "Python for Beginners", "Ajak D. Garang")  # Create the first book.
# Create a reference book to demonstrate inheritance and polymorphism.
book2 = ReferenceBook("9780000002", "Data Science Handbook", "M. Analyst")  # Create the reference book.
# Create physical copies for the normal book.
copy1 = BookCopy("CP001", book1)  # Create the first copy.
copy2 = BookCopy("CP002", book1)  # Create the second copy.
# Create a student who can borrow books.
student1 = Student("Deng Guttier", "M25B38/031", "+256-700-000-000")  # Create a student object.
# Display book details using the same method name.
print(book1.get_details())  # Call the Book version of get_details().
print(book2.get_details())  # Call the overridden ReferenceBook version of get_details().
# Display the student and the physical copies.
print(student1)  # Display the student details.
print(copy1)  # Display the first copy's status.
print(copy2)  # Display the second copy's status.


## QN.3 – Borrow and return a book

The borrowing function checks the copy before creating a loan. The return function records the return date and calculates the fine.

In [ ]:
# Borrow the first copy on 1 September 2026.
loan1 = borrow_book(student1, copy1, date(2026, 9, 1), date(2026, 9, 7))  # Borrow the book with a due date.
# Show the copy status after borrowing.
print(copy1)  # The copy should now show Borrowed.
# Return the book on 10 September 2026, three days after the due date.
fine1 = return_book(loan1, date(2026, 9, 10))  # Return the book and calculate the fine.
# Show the copy status after returning.
print(copy1)  # The copy should now show Available.
# Display the loan record and the calculated fine.
print(loan1)  # Display the loan information.
print(f"Calculated fine: UGX {fine1:,}")  # Display the final fine.


## QN.4 – Demonstrate the four data structures

- A **list** stores loans and book copies.
- A **tuple** stores fixed library rules.
- A **dictionary** stores the book catalogue.
- A **set** stores unique copy IDs.

In [ ]:
# LIST: Get every loan belonging to the student.
student_loans = student1.list_loans()  # Return the student's loans as a list.
print("Student loans:", student_loans)  # Display the list of loans.
# TUPLE: Read the fixed loan period and fine amount.
loan_period_days, fine_per_day = LIBRARY_RULES  # Unpack the two values stored in the tuple.
print("Loan period in days:", loan_period_days)  # Display the loan period.
print("Fine per late day: UGX", fine_per_day)  # Display the daily fine.
# DICTIONARY: Display all books registered in the catalogue.
print("Catalogue ISBN keys:", list(catalogue.keys()))  # Show ISBNs used as dictionary keys.
print("All book titles:", Book.all_titles())  # Display book titles from the catalogue.
# SET: Display unique physical copy IDs.
print("Unique copy IDs:", registered_copy_ids)  # Show all registered copy IDs.


## QN.5 – Validation errors

The following examples intentionally try invalid values. The `try` and `except` blocks allow the program to display a helpful error instead of stopping unexpectedly.

In [ ]:
# Try to create a book with an invalid ISBN.
try:  # Start a block where a validation error may occur.
    invalid_book = Book("ABC", "Invalid Book", "Unknown Author")  # This ISBN is not 10 or 13 digits.
except (ValueError, TypeError) as error:  # Catch the expected validation error.
    print("Book validation error:", error)  # Display the reason the value was rejected.
# Try to borrow the same copy again after it has already been returned? This copy is available now.
# Borrow it first so the next attempt can demonstrate the unavailable-copy error.
loan2 = borrow_book(student1, copy1, date(2026, 9, 11), date(2026, 9, 25))  # Borrow the available copy.
try:  # Try to borrow the same physical copy a second time.
    another_loan = borrow_book(student1, copy1, date(2026, 9, 12), date(2026, 9, 26))  # This should fail.
except ValueError as error:  # Catch the unavailable-copy error.
    print("Borrowing validation error:", error)  # Display the reason borrowing was rejected.
# Try to create a physical copy with an ID that already exists.
try:  # Start another validation test.
    duplicate_copy = BookCopy("CP001", book1)  # CP001 is already registered.
except ValueError as error:  # Catch the duplicate-ID error.
    print("Copy ID validation error:", error)  # Display the reason the copy was rejected.


## QN.6 – Summary of what the program demonstrates

1. **Encapsulation:** `Book.__isbn` is private and is accessed through the `isbn` property.
2. **Inheritance:** `Book` inherits from `LibraryItem`; `ReferenceBook` inherits from `Book`; `Student` inherits from `Person`.
3. **Abstraction:** `LibraryItem` and `Person` are abstract classes with abstract methods.
4. **Polymorphism:** `Book` and `ReferenceBook` implement `get_details()` differently.
5. **Validation errors:** Invalid ISBNs, duplicate ISBNs/copy IDs, invalid dates, invalid object types, unavailable copies, and empty required fields are rejected.
6. **Data structures:** Lists, tuples, dictionaries, and sets each have a practical purpose in the library system.

Run the code cells from top to bottom so that the classes are defined before the examples use them.